# Phase 3: Adversarial High-Frequency Detail Synthesis (Detail GAN)

**Objective:** Train an adversarial synthesis network to generate 512×512 UV displacement maps containing subject-specific micro-wrinkles and pore texture, conditioned on multi-view features and coarse neutral geometry.

### Architectural & Training Recipe:
- **U-Net Generator:** `InstanceNorm2d(affine=True)` (eliminating batch-size-4 noise and cross-sample coupling), multi-view cross-attention bottleneck with `LayerNorm` and residual connection (`q = q + MultiViewAttention(LN(q), per_view_feats)`), AdaIN style modulation from identity/expression codes, and bilinear upsampling.
- **PatchGAN Discriminator:** Spectral Normalization with 70×70 receptive field patches.
- **Lazy R1 Gradient Penalty:** Computed every 16 steps strictly in **FP32** (`torch.amp.autocast('cuda', enabled=False)`) accumulating into discriminator gradients.
- **Masked Reconstruction Loss:** L1 loss restricted strictly to valid facial UV pixels (`mask == 1.0`), annealed linearly from $\lambda = 100$ down to $\lambda = 10$ over 50,000 steps.
- **Hardware & Distributed Strategy:** 2× NVIDIA Tesla T4 GPUs via PyTorch DDP (`torchrun --nproc_per_node=2`), PyTorch 2.4+ AMP mixed-precision, and 11.5-hour emergency checkpoint saving.
- **Weights:** Continuous EMA averaging (`ema_decay = 0.999`) with buffer copying; production serialization of `ema_generator.pt`.

In [ ]:
# ── CELL 1: Environment Check & Safe Kernel Restart ─────────────────────────────
import sys
import subprocess
import os
import numpy as np

print(f"[Setup] Active NumPy version: {np.__version__}")
if np.__version__.startswith("2."):
    print("Detected NumPy 2.x. Downgrading to 1.26.4 for C-extension ABI stability...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy==1.26.4", "--quiet"])
    print("Restarting kernel...")
    os.kill(os.getpid(), 9)
else:
    print("NumPy is compatible. Proceeding.")

In [ ]:
# ── CELL 2: Install Repository Dependencies & nvdiffrast ────────────────────────
!pip install -r requirements.txt --quiet
!pip install git+https://github.com/NVlabs/nvdiffrast.git --no-build-isolation --quiet

import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "CUDA GPU accelerator required for GAN training!"
n_gpus = torch.cuda.device_count()
print(f"Detected {n_gpus} GPU(s):")
for i in range(n_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# ── CELL 3: Dataset Discovery & Normalization Stats Verification ────────────────
from pathlib import Path
import json

# Dynamically locate UV displacement dataset anywhere in /kaggle/input or local outputs
DATA_DIR = None
for stats_candidate in list(Path("/kaggle/input").glob("**/normalization_stats.json")) + list(Path(".").glob("**/normalization_stats.json")):
    DATA_DIR = stats_candidate.parent
    break

if DATA_DIR is None:
    for cand in [Path("/kaggle/input/uv_displacement_dataset_1024"), Path("/kaggle/working/uv_displacement_dataset_1024"), Path("./outputs/uv_displacement_dataset_1024"), Path("./data/uv_displacement_1024")]:
        if cand.exists():
            DATA_DIR = cand
            break

if DATA_DIR is None:
    DATA_DIR = Path("./outputs/uv_displacement_dataset_1024")

print(f"Checking dataset at: {DATA_DIR.resolve()}")
disp_files = list(DATA_DIR.glob("*_disp.png"))
print(f"Displacement maps found: {len(disp_files)}")

stats_file = DATA_DIR / "normalization_stats.json"
if stats_file.exists():
    with open(stats_file) as f:
        stats = json.load(f)
    print(f"Loaded normalization statistics: p99 = {stats.get('p99_mm', stats.get('p99_displacement_mm', 'N/A'))} mm, resolution = {stats.get('resolution', 1024)}")
else:
    print("Warning: normalization_stats.json not found in dataset directory!")


In [ ]:
# ── CELL 4: Launch Multi-GPU DDP Training on 2x T4 ─────────────────────────────
import torch
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
!torchrun --nproc_per_node={n_gpus} src/stage3_detail/trainer.py \
    --data_dir {DATA_DIR} \
    --checkpoint_dir checkpoints/stage3_detail \
    --total_steps 50000 \
    --batch_size 12

In [ ]:
# ── CELL 5: Validation Evaluation & Batch Output Diversity Gate ─────────────────
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from src.stage3_detail.generator import DetailGenerator
from src.stage3_detail.data import UVDisplacementDataset

ckpt_ema = Path('checkpoints/stage3_detail/ema_generator.pt')
if ckpt_ema.exists() and DATA_DIR.exists() and len(disp_files) > 0:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    gen = DetailGenerator().to(device)
    gen.load_state_dict(torch.load(ckpt_ema, map_location=device))
    gen.eval()
    
    val_ds = UVDisplacementDataset(str(DATA_DIR), is_train=False)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=4, shuffle=False)
    
    with torch.no_grad():
        sample = next(iter(val_loader))
        pos = sample['pos'].to(device)
        norm = sample['norm'].to(device)
        feats = sample['per_view_feats'].to(device)
        beta = sample['beta'].to(device)
        psi = sample['psi'].to(device)
        
        pred_disp = gen(pos, norm, feats, beta, psi)
        batch_std = float(pred_disp.std().item())
        
        print(f"\n--- Phase 3 Detail GAN Evaluation ---")
        print(f"Output Shape: {pred_disp.shape}")
        print(f"Displacement Min: {pred_disp.min().item():.4f} | Max: {pred_disp.max().item():.4f}")
        print(f"Batch Standard Deviation: {batch_std:.4f} (Gate: > 0.010)")
        
        # Gate Verification
        assert batch_std > 0.01, f"FAIL: Mode collapse detected (std={batch_std:.4f} <= 0.010)"
        print("SUCCESS: Detail generator exhibits healthy spatial diversity (no mode collapse).")
        
        # Visualize sample
        disp_np = pred_disp[0, 0].cpu().numpy()
        plt.figure(figsize=(6, 6))
        plt.title(f"Synthesized UV Displacement (std={batch_std:.4f})")
        plt.imshow(disp_np, cmap='inferno', vmin=-1.0, vmax=1.0)
        plt.colorbar(label='Normalized displacement [-1, 1]')
        plt.axis('off')
        plt.tight_layout()
        plt.savefig('checkpoints/stage3_detail/validation_preview.png', dpi=150)
        plt.show()
else:
    print("Checkpoint or dataset not found for local preview.")

In [ ]:
# ── CELL 6: Checkpoint Upload to Kaggle Models Registry ────────────────────────
# Ensure normalization_stats.json is copied to checkpoint_dir before upload
!cp {DATA_DIR}/normalization_stats.json checkpoints/stage3_detail/ 2>/dev/null || true

!python scripts/upload_to_kaggle_models.py \
    --checkpoint_dir checkpoints/stage3_detail \
    --handle nightshowdown/face-geo-stage3-detail-gan/pytorch/v1 \
    --version_notes "Stage 3 Detail GAN with InstanceNorm, residual cross-attention, and EMA weights" \
    --stage 3